# Day E5 — 탐색 분석과 교란 통제: 회귀 보정, 성향점수, 민감도 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E5.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — Table 1과 SMD

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(23)
n = 4000
age = rng.normal(65, 13, n)
sofa = np.clip(rng.poisson(5, n) + (age > 75), 0, 20)
lactate = np.exp(0.3 + 0.08 * sofa + rng.normal(0, 0.3, n))
p_treat = 1 / (1 + np.exp(-(-3.0 + 0.35 * sofa + 0.3 * np.log(lactate) + 0.01 * (age - 65))))
treat = rng.random(n) < p_treat
logit_death = -4.2 + 0.30 * sofa + 0.03 * (age - 65) + 0.5 * np.log(lactate) - 0.40 * treat
died = rng.random(n) < 1 / (1 + np.exp(-logit_death))
df = pd.DataFrame({"age": age, "sofa": sofa, "log_lactate": np.log(lactate),
                   "treat": treat.astype(int), "died": died.astype(int)})
covs = ["age", "sofa", "log_lactate"]

def smd(d, w=None):
    w = np.ones(len(d)) if w is None else w
    out = {}
    for c in covs:
        t, u = d.treat == 1, d.treat == 0
        m1, m0 = np.average(d[c][t], weights=w[t]), np.average(d[c][u], weights=w[u])
        v1 = np.average((d[c][t] - m1) ** 2, weights=w[t]); v0 = np.average((d[c][u] - m0) ** 2, weights=w[u])
        out[c] = round(float((m1 - m0) / np.sqrt((v1 + v0) / 2)), 3)
    return out

print(df.groupby("treat")[covs + ["died"]].mean().round(3))
print("SMD(보정 전):", smd(df))

### 예시 2 — 단순 비교 vs 회귀 보정

In [ ]:
X0 = sm.add_constant(df[["treat"]]); X1 = sm.add_constant(df[["treat"] + covs])
or_crude = np.exp(sm.Logit(df.died, X0).fit(disp=0).params["treat"])
fit = sm.Logit(df.died, X1).fit(disp=0)
ci = np.exp(fit.conf_int().loc["treat"])
print(f"단순 OR {or_crude:.2f} | 보정 OR {np.exp(fit.params['treat']):.2f} (95% CI {ci[0]:.2f}–{ci[1]:.2f}) | 진짜 조건부 OR {np.exp(-0.40):.2f}")

### 예시 3 — 성향점수 매칭과 caliper 바꾸기

In [ ]:
ps = LogisticRegression(C=1e6, max_iter=1000).fit(df[covs], df.treat).predict_proba(df[covs])[:, 1]
df["ps"] = ps
df["logit_ps"] = np.log(ps / (1 - ps))
print("성향점수 범위  치료군 %.2f–%.2f, 비치료군 %.2f–%.2f" %
      (ps[treat].min(), ps[treat].max(), ps[~treat].min(), ps[~treat].max()))

def match(d, caliper_sd):
    cal = caliper_sd * d.logit_ps.std()
    t = d[d.treat == 1].sample(frac=1, random_state=0)
    pool = d[d.treat == 0].copy()
    pairs = []
    for i, r in t.iterrows():
        if pool.empty: break
        dist = (pool.logit_ps - r.logit_ps).abs()
        j = dist.idxmin()
        if dist[j] <= cal:
            pairs.append((i, j)); pool = pool.drop(j)
    idx_t, idx_c = zip(*pairs)
    return d.loc[list(idx_t) + list(idx_c)]

for cal in [0.05, 0.2, 1.0]:
    m = match(df, cal)
    rd = m[m.treat == 1].died.mean() - m[m.treat == 0].died.mean()
    print(f"caliper {cal:>4} SD: 짝 {len(m)//2}쌍, 위험차 {rd:+.3f}, 매칭 후 SMD {smd(m.reset_index(drop=True))}")

### 예시 4 — 역확률 가중, 추정 대상의 차이, E-value

In [ ]:
p_t = df.treat.mean()
w = np.where(df.treat == 1, p_t / df.ps, (1 - p_t) / (1 - df.ps))      # 안정화 가중치
print("안정화 가중치 범위 %.2f–%.2f, 가중 후 SMD %s" % (w.min(), w.max(), smd(df, w)))
rd_w = (np.average(df.died[df.treat == 1], weights=w[df.treat == 1])
        - np.average(df.died[df.treat == 0], weights=w[df.treat == 0]))
print(f"IPW 위험차 {rd_w:+.3f} | 단순 위험차 {df.died[df.treat==1].mean()-df.died[df.treat==0].mean():+.3f}")

# 진짜 평균 효과(시뮬레이션이라 계산 가능): 모두 치료 vs 모두 비치료
base = -4.2 + 0.30 * sofa + 0.03 * (age - 65) + 0.5 * np.log(lactate)
true_rd = (1 / (1 + np.exp(-(base - 0.40)))).mean() - (1 / (1 + np.exp(-base))).mean()
att = ((1 / (1 + np.exp(-(base - 0.40)))) - (1 / (1 + np.exp(-base))))[treat].mean()
print(f"진짜 평균 위험차: 전체 인구 {true_rd:+.3f}, 치료받은 사람들 {att:+.3f}")

# E-value: 관찰된 위험비를 없애려면 측정 안 된 교란이 얼마나 강해야 하나
rr = 0.80
rr_inv = 1 / rr
print(f"\n관찰 RR {rr} 의 E-value = {rr_inv + np.sqrt(rr_inv * (rr_inv - 1)):.2f}")

## 직접 해보기

1. `notebooks/advanced/E/E5_confounding.ipynb`에서 예시 1–4를 순서대로 실행합니다.
2. Love plot(보정 전후 SMD 그림)을 그립니다. 예시 4 다음 셀에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("../../../figures/advanced/E"); out.mkdir(parents=True, exist_ok=True)
m02 = match(df, 0.2).reset_index(drop=True)
rows = pd.DataFrame({"before": smd(df), "matched (0.2)": smd(m02), "IPW": smd(df, w)})
plt.figure(figsize=(5.5, 3))
for col, mk in zip(rows.columns, ["o", "s", "^"]):
    plt.scatter(rows[col].abs(), rows.index, marker=mk, label=col)
plt.axvline(0.1, color="grey", ls="--")
plt.xlabel("|standardized mean difference|")
plt.legend()
plt.tight_layout()
plt.savefig(out / "E5_love_plot.png", dpi=150)
plt.show()